# Exploratory Data Analysis: 100,000-Record Placement Benchmark
### Statistical Distributions, Feature Interactions & Placement Drivers

---
## 📌 Overview
This notebook conducts a comprehensive Exploratory Data Analysis (EDA) on the 100,000-record campus placement benchmark dataset. We analyze academic indicators, technical proficiencies, project portfolios, and work experience distributions, exploring how these attributes interact to influence hiring outcomes and compensation packages.

## 1. Setup & Environment Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('crest')
pd.set_option('display.max_columns', None)

## 2. Dataset Ingestion & Schema Inspection

In [ ]:
data_path = '../data/raw/student_placement_data.csv'
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 3. Data Integrity & Summary Statistics

In [ ]:
print("Missing Values Summary:")
print(df.isnull().sum())

unplaced_count = (df['placement_status'] == 0).sum()
salary_nans = df['salary_lpa'].isnull().sum()
print(f"\nUnplaced Students: {unplaced_count:,} | Salary NaN count: {salary_nans:,}")
assert unplaced_count == salary_nans, "Discrepancy in unplaced salary records!"

print("\nSummary Statistics (Key Numerical Features):")
df[['cpi', 'coding_score', 'dsa_score', 'dsa_problems_solved', 'projects_count', 'salary_lpa']].describe().round(2)

## 4. Univariate Distributions: Academic & Technical Signals

In [ ]:
num_features = ['cpi', 'attendance_rate', 'coding_score', 'dsa_score', 'dsa_problems_solved', 'aptitude_score']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#1f77b4', bins=30)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Bivariate Analysis: Determinants of Placement & CTC Packages

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# CPI vs Placement
sns.boxplot(data=df, x='placement_status', y='cpi', ax=axes[0], palette=['#e7298a', '#1b9e77'])
axes[0].set_title('CPI Distribution by Placement Status', fontweight='bold')
axes[0].set_xticklabels(['Unplaced (0)', 'Placed (1)'])

# Coding Score vs Placement
sns.boxplot(data=df, x='placement_status', y='coding_score', ax=axes[1], palette=['#e7298a', '#1b9e77'])
axes[1].set_title('Coding Score by Placement Status', fontweight='bold')
axes[1].set_xticklabels(['Unplaced (0)', 'Placed (1)'])

# Backlogs vs Placement Probability
backlog_rates = df.groupby('backlogs')['placement_status'].mean() * 100
sns.barplot(x=backlog_rates.index, y=backlog_rates.values, ax=axes[2], color='#2b5c8f')
axes[2].set_title('Placement Rate (%) by Active Backlogs', fontweight='bold')
axes[2].set_ylabel('Placement Rate (%)')
axes[2].set_xlabel('Backlogs Count')

plt.tight_layout()
plt.show()

## 6. Multivariate Correlation Matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['salary_lpa'])
corr = numeric_cols.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap Across Numerical Attributes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Salary Package Analysis for Placed Graduates

In [ ]:
placed_df = df[df['placement_status'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(placed_df['salary_lpa'], kde=True, ax=axes[0], color='#2ca02c', bins=40)
axes[0].set_title('Placed CTC Package Distribution (LPA)', fontweight='bold')
axes[0].set_xlabel('Annual CTC (LPA)')

sns.scatterplot(data=placed_df, x='coding_score', y='salary_lpa', alpha=0.3, ax=axes[1], color='#1f77b4')
axes[1].set_title('Coding Score vs Salary LPA (Placed Students)', fontweight='bold')
axes[1].set_xlabel('Coding Score')
axes[1].set_ylabel('Salary LPA')

plt.tight_layout()
plt.show()